In [1]:
# import library
import numpy as np
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — DATA KOMPONEN MURNI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════

pure_components = {
    # urutan ikutin kolom
    
    #   nama                 m        σ/Å      ε/K
    "glycine"      : (4.8495, 2.327, 216.96),
    "alanine"      : (5.4647, 2.5222, 287.59),
    "valine"       : (7.4851, 2.5888, 306.41),
    "leucine"      : (8.3037, 2.7, 330),
    "serine"       : (7.0236, 2.284, 236.92),
    # kalo nambah jangan lupa koma
}

# Urutan molekul — harus SAMA dengan urutan kolom di S_terms (SECTION 4)
# molecules = ["isobutyl", "isovaleric", "isocaproic"]
molecules = ["glycine", "alanine", "valine", "leucine", "serine"]


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — PARAMETER REFERENSI GRUP (INPUT — SUDAH DIKETAHUI)
# ══════════════════════════════════════════════════════════════════════════════

GC_ref = {
    #   nama        m        σ/Å      ε/K
    "CH3"  : (0.378463,  4.46893,  160.224  ),
    "CH2"  : (0.314747,  4.24947,  273.852  ),
    "CH"   : (-1.597,    2.66463,  -37.2512 ),
    "NH2"  : (2.63534,   2.48521,  167.534  ),
    "OH"   : (2.60111,   2.42260,  164.943  ),
    "COOH" : (0.552166,  2.858382385,  -646.5938921)
    #tambahkan baru kalo ada jangan lupa koma
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — PARAMETER PERTURBATIF μ YANG SUDAH DIKETAHUI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════
# unknown masukin di section 5
# masukin nilai yang sudah diitung me dan ms3 nya
# Format: "GRUP1-GRUP2" : {"m": nilai, "me": nilai, "ms3": nilai}

mu_known = {
    "CH3-CH3"  : {"m":  0.431978,  "me":  94.02534772,  "ms3":  1.364177384},
    "CH3-CH"   : {"m":  0.238592,  "me":  -5.474972039,  "ms3":  7.579641756},
    
    "CH2-CH3"  : {"m":  0.220392,  "me":  54.25616028,  "ms3":  0.723277646 },
    "CH2-CH2"  : {"m":  0.0,  "me":  0.0,  "ms3":  0.0 },
    "CH2-CH"   : {"m":  0.068613,  "me": -51.44046416,  "ms3":  5.286545478 },
    
    "CH-CH"    : {"m":  0.42962,  "me": 145.976161,  "ms3":  -5.310938 },

    "CH3-NH2"  : {"m": -0.252981,  "me":  12.1086618,   "ms3": -8.213189111 },
    "CH2-NH2"  : {"m": -0.55194,   "me": -59.75165212,  "ms3": -8.08211105 },
    "CH-NH2"   : {"m": 0.113279,   "me": -28.964268,  "ms3": 8.495794 },
    
    "CH3-OH"   : {"m": -0.716115,   "me": -103.608394,  "ms3": -9.948668064 },
    "CH2-OH"   : {"m": -0.454731,   "me": -20.00948549,  "ms3": -7.033520356 },
    "CH-OH"    : {"m": 0.698168,   "me": 89.666537,  "ms3": 7.926724 },
    
    "CH3-COOH" : {"m" : 0.921519,  "me" : 403.721118,  "ms3" : 12.889752 },
    "CH2-COOH" : {"m" : 0.627864, "me" : 357.294354, "ms3" : 14.908606 },
    "CH-COOH"  : {"m" : 0.5497, "me" : 178.204895, "ms3" : 11.758071 },

    "NH2-OH"   : {"m" : -0.937732, "me" : -570.804721, "ms3" : -34.30531 },
    
    #kalo nambah baris baru jangan lupa koma
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — S-TERM (INPUT MANUAL)
# ══════════════════════════════════════════════════════════════════════════════


# Urutan kolom harus SAMA dengan urutan di variabel 'molecules' (SECTION 1).
# Jika pasangan tidak ada dalam molekul isi 0.0

S_terms = {
    # urutan kolom sesuai ama urutan input molecules
    # penamaan pasangan harus sama ama mu_known
    # HATI HATI PLS AMA YG ADA CH2 JGN KEBALIK
    # 
    #             ["glycine", "alanine", "valine", "leucine", "serine"]
    

    "CH3-CH3" : [0, 0, 0.5, 0.5, 0],
    "CH3-CH" : [0, 1, 3, 2.66666666666667, 0],
    "CH2-CH3" : [0, 0, 0, 1, 0],
    
    "CH2-CH2" : [0, 0, 0, 0, 0],
    "CH2-CH" : [0, 0, 0, 2, 1],

    "CH-CH" : [0, 0, 1, 0.5, 0],

    "CH3-NH2" : [0, 0.5, 0.666666666666667, 0.5, 0],
    "CH2-NH2" : [1, 0, 0, 0.5, 0.5],
    "CH-NH2" : [0, 1, 1.5, 1.33333333333333, 1],
 
    "CH3-OH" : [0, 0, 0, 0, 0],
    "CH2-OH" : [0, 0, 0, 0, 1],
    "CH-OH" : [0, 0, 0, 0, 0.5],
 
    "CH3-COOH" : [0, 0.5, 0.666666666666667, 0.5, 0],
    "CH2-COOH" : [1, 0, 0, 0.5, 0.5],
    "CH-COOH" : [0, 1, 1.5, 1.33333333333333, 1],
 
    "NH2-OH" : [0, 0, 0, 0, 0.333333333333333],
    "NH2-COOH" : [0.5, 0.5, 0.5, 0.5, 0.5],
    "COOH-OH" : [0, 0, 0, 0, 0.333333333333333],


    # kalo nambah jangan lupa koma
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — VARIABEL YANG DI-FITTING (UNKNOWN)
# ══════════════════════════════════════════════════════════════════════════════
# Daftar pasangan μ yang BELUM diketahui dan akan di-fitting.
# Nama harus konsisten dengan yang akan dipakai di SECTION 6.
# Urutan di sini menentukan urutan parameter dalam vektor x optimizer.
#
# Untuk setiap properti (m, me, ms3), ada 1 nilai per pasangan unknown.
# Total unknown = len(unknown_pairs) × 3

unknown_pairs = [
    "NH2-COOH",
    "COOH-OH"
    
    
    #"COOH-ref",     π_COOH (parameter referensi grup COOH)
    #"CH3-COOH",     # μ(CH3-COOH) — perturbasi CH3 terhadap COOH
    #kalo nambah jangan lupa koma
]

# Jumlah unknown per properti
N_unk = len(unknown_pairs)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — PERSAMAAN MODEL PER MOLEKUL
# ══════════════════════════════════════════════════════════════════════════════
# Di sinilah Anda menulis persamaan π_GC untuk setiap molekul.
# Fungsi ini dipanggil oleh optimizer untuk menghitung prediksi GC.
#
# CARA PENGGUNAAN:
#   gc(grup, prop)          → nilai π referensi grup (dari SECTION 2)
#   mk(pasangan, prop)      → nilai μ known (dari SECTION 3)
#   S(pasangan, mol_idx)    → nilai S-term (dari SECTION 4)
#   unk[i]                  → nilai unknown ke-i (dari SECTION 5, urutan sama)
#
# KOEFISIEN (1 atau 2 di depan μ·S):
#   Gunakan 2× jika pasangan SIMETRIS dalam molekul (misalnya 2 CH3 yang
#   masing-masing berinteraksi dengan COOH dari dua arah yang setara).
#   Gunakan 1× jika tidak simetris atau hanya ada 1 pasang.
#   Lihat komentar pada masing-masing molekul di bawah.
#
# STRUKTUR: mol_idx adalah indeks molekul (0=acetic, 1=propionic, dst.)

def model(mol_idx, prop, unk):
    #spesifikin unknown here
    mu_NH2_COOH = unk[0]
    mu_COOH_OH = unk[1]
    
    """
    mol_idx : int   → indeks molekul (urutan sesuai 'molecules')
    prop    : str   → "m", "me", atau "ms3"
    unk     : list  → nilai unknown saat ini [unk[0], unk[1], unk[2], ...]
    """
    
    if mol_idx == 0:
        #glycine
        return (
            0 * gc("CH3", prop)
            + 1 * gc("CH2", prop)
            + 0 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 0 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 2 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 2 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 2 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 2 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 2 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 2 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 2 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 2 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )
        
        
    elif mol_idx == 1:
        # alanine
         return (
            1 * gc("CH3", prop)
            + 0 * gc("CH2", prop)
            + 1 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 0 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 2 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 2 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 2 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 2 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 2 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 2 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 2 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 2 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )

    elif mol_idx == 2:
        # valine
         return (
            2 * gc("CH3", prop)
            + 0 * gc("CH2", prop)
            + 2 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 0 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 2 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 2 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 2 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 2 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 2 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 2 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 2 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 2 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )

    elif mol_idx == 3:
        # leucine
         return (
            2 * gc("CH3", prop)
            + 1 * gc("CH2", prop)
            + 2 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 0 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 2 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 2 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 2 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 2 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 2 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 2 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 2 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 2 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )

    elif mol_idx == 4:
        # serine
         return (
            0 * gc("CH3", prop)
            + 1 * gc("CH2", prop)
            + 2 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 1 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 2 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 2 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 2 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 2 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 2 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 2 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 2 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 2 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )

            

    else:
        raise ValueError(f"mol_idx={mol_idx} tidak ada dalam daftar molekul.")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — BATAS (BOUNDS) UNTUK UNKNOWN
# ══════════════════════════════════════════════════════════════════════════════
# Urutan: untuk setiap properti (m, me, ms3), untuk setiap unknown
# Total batas = N_unk × 3 (m dulu, lalu me, lalu ms3)
#
# Format: (batas_bawah, batas_atas)
# Jika setelah running ada parameter yang menempel di batas → perlebar batas.
#
# Panduan fisik:
#   m_grup      : biasanya 0.1–30 (positif, tapi COOH bisa negatif jika CH negatif)
#   μ_m         : ±50 cukup lebar
#   μ_mε (K)    : ±2000 K
#   μ_mσ³ (Å³) : ±500

# row: urutan pi atau perturbansi (m, me, ms3)
# column: urutan unknown

bounds = [
    # μm(CH-CH)
    (-10.0, 10.0), (-10.0, 10.0),

    # μme(CH-CH)
    (-2000.0, 2000.0), (-2000.0, 2000.0),

    # μms3(CH-CH)
    (-500.0, 500.0), (-500.0, 500.0),
]

#jangan lupa koma di akhir kurung, dan dalam bentuk float

# Validasi: jumlah bounds harus = N_unk × 3
assert len(bounds) == N_unk * 3, (
    f"Jumlah bounds ({len(bounds)}) harus = N_unk×3 = {N_unk*3}. "
    f"Sesuaikan SECTION 7 dengan jumlah unknown di SECTION 5."
)


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN OTOMATIS — TIDAK PERLU DIEDIT
# (Derived quantities, objective function, optimizer, output)
# ══════════════════════════════════════════════════════════════════════════════

# ── Helper: konversi (m, σ, ε) → (m, mε, mσ³) ──────────────────────────────
def _pi_triple(m, sigma, eps):
    return {"m": m, "me": m * eps, "ms3": m * sigma**3}

# ── Helper: ambil π referensi grup ──────────────────────────────────────────
def gc(group, prop):
    """Kembalikan nilai π (m, mε, atau mσ³) untuk grup referensi."""
    m_g, sig_g, eps_g = GC_ref[group]
    return _pi_triple(m_g, sig_g, eps_g)[prop]

# ── Helper: ambil μ known ────────────────────────────────────────────────────
def mk(pair, prop):
    """Kembalikan nilai μ yang sudah diketahui untuk pasangan dan properti."""
    if pair not in mu_known:
        raise KeyError(
            f"Pasangan '{pair}' tidak ada di mu_known (SECTION 3). "
            f"Tambahkan jika diketahui, atau pindahkan ke unknown_pairs (SECTION 5)."
        )
    return mu_known[pair][prop]

# ── Helper: ambil S-term ─────────────────────────────────────────────────────
def S(pair, mol_idx):
    """Kembalikan nilai S-term untuk pasangan dan indeks molekul."""
    if pair not in S_terms:
        raise KeyError(
            f"Pasangan '{pair}' tidak ada di S_terms (SECTION 4). "
            f"Tambahkan nilainya."
        )
    return S_terms[pair][mol_idx]

# ── Hitung target π dari data komponen murni ─────────────────────────────────
targets = {prop: [] for prop in ("m", "me", "ms3")}
for mol in molecules:
    m, sig, eps = pure_components[mol]
    pi = _pi_triple(m, sig, eps)
    for prop in ("m", "me", "ms3"):
        targets[prop].append(pi[prop])
targets = {prop: np.array(v) for prop, v in targets.items()}

# ── Unpack unknown dari vektor x ─────────────────────────────────────────────
def unpack(x, prop):
    """Ambil slice unknown untuk properti tertentu dari vektor x optimizer."""
    i = {"m": 0, "me": 1, "ms3": 2}[prop]
    return x[i * N_unk : (i + 1) * N_unk]

# ── Objective function ────────────────────────────────────────────────────────
def objective(x):
    """
    F_OBJ = Σ_prop Σ_molekul [ (π_target - π_GC) / π_target ]²

    Vektor x berisi semua unknown untuk semua properti:
      x[0 : N_unk]           → unknown untuk m
      x[N_unk : 2*N_unk]     → unknown untuk mε
      x[2*N_unk : 3*N_unk]   → unknown untuk mσ³
    """
    F = 0.0
    for prop in ("m", "me", "ms3"):
        unk = unpack(x, prop)
        for mol_idx in range(len(molecules)):
            pi_GC  = model(mol_idx, prop, unk)
            pi_tgt = targets[prop][mol_idx]
            F += ((pi_tgt - pi_GC) / pi_tgt) ** 2
    return F

# ── Optimasi dua tahap ───────────────────────────────────────────────────────
print("=" * 62)
print("  PC-SAFT PertGC — Universal Fitting Template")
print("=" * 62)
print(f"\n  Molekul  : {', '.join(molecules)}")
print(f"  Unknown  : {', '.join(unknown_pairs)}")
print(f"  Total    : {N_unk} unknown × 3 properti = {N_unk*3} parameter\n")

print("  Stage 1: Global search (differential_evolution)...")
res_g = differential_evolution(
    objective, bounds,
    seed=42, maxiter=8000, tol=1e-14,
    popsize=25, mutation=(0.5, 1.5), recombination=0.9,
    polish=True, disp=False,
)

print("  Stage 2: Local refinement (L-BFGS-B)...")
res_l = minimize(
    objective, res_g.x,
    method="L-BFGS-B", bounds=bounds,
    options={"maxiter": 100000, "ftol": 1e-16, "gtol": 1e-13},
)

x_opt = res_l.x if res_l.fun < res_g.fun else res_g.x
F_opt = min(res_l.fun, res_g.fun)

# ── Cetak hasil ──────────────────────────────────────────────────────────────
prop_label = {"m": "m", "me": "m·ε  (K)", "ms3": "m·σ³ (Å³)"}

print("\n" + "=" * 62)
print("  FITTED PARAMETERS")
print("=" * 62)
for prop in ("m", "me", "ms3"):
    unk = unpack(x_opt, prop)
    print(f"\n  Properti: π = {prop_label[prop]}")
    for j, pair in enumerate(unknown_pairs):
        print(f"    {pair:<22} = {unk[j]:>16.6f}")

print(f"\n  F_OBJ (final) = {F_opt:.6e}")

# ── Validasi ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  VALIDATION — Target vs. GC-Predicted (APD%)")
print("=" * 62)

for prop in ("m", "me", "ms3"):
    unk = unpack(x_opt, prop)
    print(f"\n  π = {prop_label[prop]}")
    print(f"  {'Molekul':<14} {'Target':>13} {'GC-Pred':>13} {'APD%':>9}")
    print(f"  {'-'*52}")
    for mol_idx, mol in enumerate(molecules):
        pi_tgt = targets[prop][mol_idx]
        pi_gc  = model(mol_idx, prop, unk)
        apd    = 100.0 * abs(pi_tgt - pi_gc) / abs(pi_tgt)
        print(f"  {mol:<14} {pi_tgt:>13.5f} {pi_gc:>13.5f} {apd:>8.4f}%")

# ── Tabel ringkasan ───────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  SUMMARY TABLE (copy-paste ready)")
print("=" * 62)
print(f"  {'Parameter':<24} {'m':>12} {'m·ε (K)':>14} {'m·σ³ (Å³)':>14}")
print("  " + "-" * 66)
for j, pair in enumerate(unknown_pairs):
    vals = [unpack(x_opt, prop)[j] for prop in ("m", "me", "ms3")]
    print(f"  {pair:<24} {vals[0]:>12.6f} {vals[1]:>14.4f} {vals[2]:>14.6f}")

# ── Diagnostik batas ──────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  DIAGNOSTIC — Bound Check")
print("=" * 62)

# Buat label untuk semua parameter (urutan: m dulu, me, ms3)
all_labels = [f"μm({p})"   for p in unknown_pairs] + \
             [f"μme({p})"  for p in unknown_pairs] + \
             [f"μms3({p})" for p in unknown_pairs]

bound_hit = False
for i, (val, (lo, hi)) in enumerate(zip(x_opt, bounds)):
    tol = max(1e-4, 1e-4 * abs(hi - lo))
    if abs(val - lo) < tol or abs(val - hi) < tol:
        print(f"  ⚠  {all_labels[i]} = {val:.6f} menempel di batas [{lo}, {hi}].")
        print(f"      → Perlebar batas ini di SECTION 7 dan jalankan ulang.")
        bound_hit = True
if not bound_hit:
    print("  ✓  Tidak ada parameter yang menempel di batas. Solusi interior.")

  PC-SAFT PertGC — Universal Fitting Template

  Molekul  : glycine, alanine, valine, leucine, serine
  Unknown  : NH2-COOH, COOH-OH
  Total    : 2 unknown × 3 properti = 6 parameter

  Stage 1: Global search (differential_evolution)...
  Stage 2: Local refinement (L-BFGS-B)...

  FITTED PARAMETERS

  Properti: π = m
    NH2-COOH               =         1.822913
    COOH-OH                =         2.274718

  Properti: π = m·ε  (K)
    NH2-COOH               =       748.581584
    COOH-OH                =       212.552467

  Properti: π = m·σ³ (Å³)
    NH2-COOH               =       -23.614413
    COOH-OH                =        45.085865

  F_OBJ (final) = 1.837483e-01

  VALIDATION — Target vs. GC-Predicted (APD%)

  π = m
  Molekul               Target       GC-Pred      APD%
  ----------------------------------------------------
  glycine              4.84950       4.84915   0.0072%
  alanine              5.46470       6.26356  14.6186%
  valine               7.48510       7.10121